# Bitcoin: label noise

Sections 2 and 3.2. Community reports are noisy. On the training partition of Scenario 2 we compute, for every wallet, the out-of-fold probability of its own label, $s_i = P(y_i \mid x_i)$, with five-fold cross-validation of the tuned Random Forest pipeline. ChainAbuse fraud wallets with $s_i < 0.3$ are treated as mislabelled and removed. The model is refit on the cleaned training partition and scored on the same test partition as Table 3.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "src"))

import json

import joblib
import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold, cross_val_predict, train_test_split

from bfx.bitcoin import FEATURES
from bfx.modeling import SEED, scores

DATA = ROOT / "data" / "bitcoin" / "scenarios"
RESULTS = ROOT / "results" / "bitcoin"
MODELS = ROOT / "models" / "bitcoin"

data = pd.read_csv(DATA / "scenario_2.csv")
train, test = train_test_split(data, test_size=0.2, stratify=data["y"], random_state=SEED)
baseline = joblib.load(MODELS / "scenario_2_RF.joblib")

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
proba = cross_val_predict(clone(baseline), train[FEATURES], train["y"], cv=cv, method="predict_proba")
train = train.assign(self_confidence=proba[np.arange(len(train)), train["y"].to_numpy()])

suspect = (train["source"] == "chainabuse") & (train["y"] == 1) & (train["self_confidence"] < 0.3)
n_fraud = int(((train["source"] == "chainabuse") & (train["y"] == 1)).sum())
print(f"flagged {suspect.sum()} of {n_fraud} ChainAbuse fraud wallets in the training partition "
      f"({suspect.sum() / n_fraud:.1%})")

In [ ]:
cleaned = clone(baseline).fit(train.loc[~suspect, FEATURES], train.loc[~suspect, "y"])

before = scores(test["y"], baseline.predict(test[FEATURES]))
after = scores(test["y"], cleaned.predict(test[FEATURES]))
comparison = pd.DataFrame({"before": before, "after": after}).loc[["precision", "recall", "f1", "accuracy"]]
comparison.round(3)

As a cross-check, the confident-joint estimate of cleanlab on the same out-of-fold probabilities:

In [ ]:
from cleanlab.filter import find_label_issues

issues = find_label_issues(train["y"].to_numpy(), proba)
print(f"cleanlab flags {issues.sum()} training wallets, "
      f"{(issues & (train['source'] == 'chainabuse').to_numpy() & (train['y'] == 1).to_numpy()).sum()} of them ChainAbuse fraud")

In [ ]:
_ = (RESULTS / "confident_learning.json").write_text(json.dumps({
    "flagged": int(suspect.sum()), "chainabuse_fraud_in_train": n_fraud,
    "f1_macro_before": before["f1"], "f1_macro_after": after["f1"],
}, indent=2, default=float))